# NORTHSTAR practical — can the data be shared?

**DASC513 · Data Engineering block · Week 6 · anonymisation and safe handling**

> ### Before you do anything else
> **File → Save a copy in Drive.** Until you do that you are looking at a read-only
> copy and your work disappears when the tab closes. Do it now, then carry on.

This notebook runs in Google Colab, or on your own machine if you have Python and
pandas. Nothing needs installing. The data loads in the next cell straight from
the course GitHub repository; if that fails (no internet, or you are working
offline), it will ask you to upload the CSV files you downloaded from Canvas.

## How to use this notebook

A notebook is a sequence of **cells**. Grey cells are code; white cells are
writing. Run a cell by clicking into it and pressing **Shift + Enter**.

Three kinds of cell in this notebook:

| Marker | What to do |
|---|---|
| **READ** | Explanation. No code to run. |
| **RUN** | Code that is already written. Read it first, predict what it will do, then run it. |
| **YOUR TURN** | Code with something missing, or a written answer to give. |

Two rules that will save you an hour of confusion:

1. **Cells run in the order you run them, not the order they appear.** If
   something breaks unexpectedly, go to *Kernel → Restart and Run All* and start
   again from the top.
2. **Read code before running it.** Say it out loud in English. `df["sex"]`
   is "the sex column of df". `df["sex"].value_counts()` is "count how many
   times each value appears in the sex column of df". If you can say it, you
   understand it.

## The enquiry loop

Every task below follows the same five steps. This is the actual method; the
pandas is just how you carry it out.

1. **Observe** — look at the data without deciding anything yet.
2. **Question** — turn what you noticed into a question that has an answer.
3. **Test** — write the smallest piece of code that answers it.
4. **Interpret** — what does the answer mean *for the research question*?
5. **Decide** — fix it, flag it, exclude it, or accept it. Write down which and why.

Step 5 is the one people skip. A data quality report full of tables and empty of
decisions is not a data quality report.

In [ ]:
# RUN. Load the data. Works in Colab, and on your own machine if the CSV files
# are beside the notebook. If neither, Colab will ask you to upload them.
import io, os
import pandas as pd

DATA_REPO = "https://raw.githubusercontent.com/civic-health-innovation-labs/dasc513-data-engineering/main/"
DATA_URL = DATA_REPO + "NORTHSTAR_redcap_export.csv"
DICT_URL = DATA_REPO + "NORTHSTAR_data_dictionary.csv"

_uploaded = {}

def load(name, url):
    if url:
        try:
            return pd.read_csv(url, dtype=str)
        except Exception as e:          # offline, or the repository moved
            print(f"Could not fetch {name} from GitHub ({type(e).__name__}); trying a local copy.")
    if os.path.exists(name):
        return pd.read_csv(name, dtype=str)
    if name not in _uploaded:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f"{name} not found. Put it next to this notebook, or check the URL above.")
        print(f"Choose {name} (from Canvas) when the button appears.")
        _uploaded.update(files.upload())
    return pd.read_csv(io.BytesIO(_uploaded[name]), dtype=str)

df = load("NORTHSTAR_redcap_export.csv", DATA_URL)
data_dictionary = load("NORTHSTAR_data_dictionary.csv", DICT_URL)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"loaded {df.shape[0]} records with {df.shape[1]} fields, and a data dictionary of {len(data_dictionary)} rows")

---
# Part 0 — Two things you need before starting

**READ.** This dataset has some rough edges that you will take apart properly in
week 7. Two of them get in the way today, so here they are dealt with up front.

**Dates.** The three hospitals write dates three different ways. The helper below
parses each site's dates using that site's format, and adds a properly parsed
admission date column. You will build this yourself in week 7; today, just run it.

**A duplicate.** Records 47 and 88 are the same person, entered twice. Week 7
shows you how to find that. Today you only need to know it is there, because it
makes a useful test of whether your pseudonymisation is consistent.

In [ ]:
# RUN. Site-specific date parsing (explained properly in week 7).
SITE_FORMATS = {
    "mersey_general": "%d/%m/%Y",
    "calderbrook":    "%Y-%m-%d",
    "wirral_park":    "%d-%b-%y",
}

def parse_dates(frame, column):
    # Parse a date column, using the right format for each site.
    out = pd.Series(pd.NaT, index=frame.index)
    for site, fmt in SITE_FORMATS.items():
        rows = frame["redcap_data_access_group"] == site
        out[rows] = pd.to_datetime(frame.loc[rows, column], format=fmt, errors="coerce")
    return out

df["adm_date_parsed"] = parse_dates(df, "adm_date")
print("admission dates parsed:", df["adm_date_parsed"].notna().sum(), "of", len(df))
df[["record_id", "redcap_data_access_group", "adm_date", "adm_date_parsed"]].head(6)

# Part 1 — What in here could identify a patient?

**READ.** Before touching anything, classify every column into one of three
groups:

- **Direct identifier** — identifies a person on its own. Name, NHS number.
- **Indirect identifier** — does not identify anyone alone, but in
  combination with others it narrows to one person. Date of birth, postcode, sex,
  admission date, a rare diagnosis.
- **Neither** — a clinical value that carries no meaningful identifying
  information.

The middle category is where re-identification actually happens, and it is the
one people underestimate.

In [ ]:
# RUN. All the columns, so you can work through them.
for i, c in enumerate(df.columns):
    if c.endswith(("_num", "_parsed")):
        continue
    print(f"{c:28}", end="   " if i % 3 != 2 else "\n")

**YOUR TURN 1.** Fill in the three lists below. Argue about the borderline ones
with the person next to you — the argument is the point.

In [ ]:
# YOUR TURN 1. Complete these lists.

DIRECT = [
    "patient_forename",
    "patient_surname",
    # ... what else?
]

INDIRECT = [
    "dob",
    "postcode",
    # ... what else? think about dates, and think about rare values
]

FREE_TEXT = [
    "clinician_comments_txt",
    # ... what else?
]

print(f"{len(DIRECT)} direct, {len(INDIRECT)} indirect, {len(FREE_TEXT)} free text")

---
# Part 2 — Applying the treatment

**READ.** Now we build a de-identified copy. Note the word **copy**: you never
overwrite the source. If you get it wrong you want to be able to go back.

We will treat each category differently.

In [ ]:
# RUN. Start from a copy, and remove the direct identifiers outright.
safe = df.copy()

to_drop = ["patient_forename", "patient_surname", "nhs_number",
           "hospital_number", "gp_practice_code"]

safe = safe.drop(columns=to_drop)
print("dropped:", to_drop)
print("columns remaining:", safe.shape[1])

**READ — but we still need to identify patients internally.**

Dropping the NHS number means you can no longer tell which rows belong to the
same person, which breaks any longitudinal analysis. The standard answer is a
**pseudonym**: a consistent, meaningless replacement.

In [ ]:
# RUN. A pseudonymous study ID, derived from the NHS number.
import hashlib

SALT = "northstar-teaching-salt-do-not-reuse"

def pseudonymise(value, salt=SALT):
    # Turn an identifier into a consistent, meaningless string.
    if not isinstance(value, str) or value.strip() == "":
        return ""
    digest = hashlib.sha256((salt + value.strip()).encode()).hexdigest()
    return "P-" + digest[:10]

safe["study_id"] = df["nhs_number"].apply(pseudonymise)

safe[["record_id", "study_id"]].head(5)

**YOUR TURN 2 — the question that matters.**

1. Records 47 and 88 are the same person (see Part 0). Check they got the same
   `study_id`. Why did that work?
2. Suppose we had left the salt out and just hashed the NHS number. An NHS number
   is 10 digits. How many possible NHS numbers are there? How long would it take
   a laptop to hash all of them and build a lookup table?
3. So what is the salt actually for, and **where must it be kept**?
4. Is `study_id` personal data? Argue it both ways.

In [ ]:
# RUN. Check the duplicate pair got the same pseudonym.
safe.loc[safe["record_id"].isin(["47", "88"]), ["record_id", "study_id"]]

In [ ]:
# RUN. Date of birth is a direct identifier. The usual treatment is to replace
# it with an age band. First we need the age.
safe["dob_parsed"] = parse_dates(df, "dob")
age_years = (df["adm_date_parsed"] - safe["dob_parsed"]).dt.days / 365.25

age_years.describe()

**YOUR TURN 2b — stop and look at that.**

Read the summary carefully before going further.

1. What is the **minimum** age? Is that possible?
2. How many patients have a negative age? Use
   `(age_years < 0).sum()` to find out.
3. Which site are they at? Try
   `df.loc[age_years < 0, "redcap_data_access_group"].value_counts()`.
4. Now look at a few of their raw dates of birth with
   `df.loc[age_years < 0, "dob"].head()`. What year does `66` become when a
   computer reads `18-Jul-66`?

This is a real and very common bug. Two-digit years are ambiguous, and Python
resolves them with a fixed cut-off: `00` to `68` become 20xx, `69` to `99`
become 19xx. For dates of birth in an adult study, that is wrong for anyone born
before 1969.

In [ ]:
# YOUR TURN 2b. Investigate. Run each of these.
print("negative ages:", (age_years < 0).sum())
print(df.loc[age_years < 0, "redcap_data_access_group"].value_counts())
df.loc[age_years < 0, ["record_id", "dob"]].head()

In [ ]:
# RUN. The fix: any date of birth in the future must be a century too late.
def fix_century(d, today=pd.Timestamp("2025-03-01")):
    if pd.isna(d):
        return d
    return d.replace(year=d.year - 100) if d > today else d

safe["dob_parsed"] = safe["dob_parsed"].apply(fix_century)
age_years = (df["adm_date_parsed"] - safe["dob_parsed"]).dt.days / 365.25

print("negative ages now:", (age_years < 0).sum())
age_years.describe()

In [ ]:
# RUN. Now the age bands.
safe["age_band"] = pd.cut(
    age_years,
    bins=[0, 18, 40, 55, 65, 75, 85, 120],
    labels=["<18", "18-39", "40-54", "55-64", "65-74", "75-84", "85+"],
)
safe = safe.drop(columns=["dob", "dob_parsed"])

print("patients with an age band:", safe["age_band"].notna().sum(), "of", len(safe))
safe["age_band"].value_counts().sort_index()

**YOUR TURN 2c.** Look at the age band counts.

- How many patients are in the smallest band? Is that a problem?
- The `<18` band has one patient — a 4-year-old, in a study of adults. What are
  the *two separate* problems there, one governance and one data quality?
- Would wider bands be safer? What would you lose?

In [ ]:
# RUN. Postcodes become districts.
# NOTE: the postcodes in this file are messy — some lowercase, some with
# no space, some with two spaces. Read this code carefully.

def postcode_district(pc):
    if not isinstance(pc, str) or pc.strip() == "":
        return ""
    clean = pc.replace(" ", "").upper()
    return clean[:-3]        # the outward code is everything except the last 3

safe["postcode_district"] = df["postcode"].apply(postcode_district)
safe = safe.drop(columns=["postcode"])

safe["postcode_district"].value_counts().head(12)

**YOUR TURN 2d — a deliberate trap.**

An obvious way to write that function would be `pc.split(" ")[0]` — "take
everything before the space".

Try it on these four values and see what happens:

`"L7 2EZ"`, `"PR80GQ"`, `"ch42 9aa"`, `" CH44  5TU "`

Then answer: why does the version above use `clean[:-3]` instead? What does that
tell you about writing cleaning code against data you have not looked at
properly?

In [ ]:
# YOUR TURN 2d. Try the naive version and compare.
test_values = ["L7 2EZ", "PR80GQ", "ch42 9aa", " CH44  5TU "]

for v in test_values:
    naive = v.split(" ")[0]
    better = postcode_district(v)
    print(f"{v!r:18}  naive: {naive!r:10}  better: {better!r}")

In [ ]:
# RUN. Dates become days since admission, so the clinical timeline survives
# but the calendar does not.
for column in ["symptom_onset_date", "outcome_date", "icu_admit_date", "steroid_start_date"]:
    parsed = parse_dates(df, column)
    safe["days_to_" + column.replace("_date", "")] = (parsed - df["adm_date_parsed"]).dt.days
    safe = safe.drop(columns=[column])

safe = safe.drop(columns=["adm_date", "adm_time", "adm_date_parsed", "outcome_date_parsed"], errors="ignore")

safe[["study_id", "days_to_symptom_onset", "days_to_outcome", "days_to_icu_admit"]].head(8)

---
# Part 3 — Is it actually anonymous now?

**READ.** We have dropped the names and the NHS number, banded the ages, cut the
postcodes back and converted the dates. It feels safe.

Feeling safe is not a method. **k-anonymity** is one way to check: for a chosen
set of indirect identifiers, every combination that appears should appear at least
*k* times. If a combination appears once, that row is unique, and anyone holding
matching information from elsewhere can pick that person out.

k=5 is a common convention in UK health data. It is a convention, not a law.

In [ ]:
# RUN. Count how many people share each combination of indirect identifiers.
def k_check(frame, indirect_identifiers, k=5):
    groups = frame.groupby(indirect_identifiers, dropna=False).size()
    unique_rows = (groups == 1).sum()
    below_k = groups[groups < k].sum()
    print(f"indirect identifiers: {indirect_identifiers}")
    print(f"  smallest group size (k)     : {groups.min()}")
    print(f"  combinations seen only once : {unique_rows}")
    print(f"  rows in groups smaller than {k}: {below_k} of {len(frame)}")
    return groups.sort_values()

# Start with just two indirect identifiers.
groups = k_check(safe, ["sex", "age_band"], k=5)

**Interpret.** With only sex and age band, 14 patients out of 100 sit in a group
smaller than 5. Not perfect, but it looks manageable — you could suppress those
rows and move on.

Now add one more field, home district, and run it again.

In [ ]:
# RUN. Add postcode district.
_ = k_check(safe, ["sex", "age_band", "postcode_district"], k=5)

**Interpret.** One extra field, and the picture collapses. This is the important
property of indirect identifiers: risk does not add up, it multiplies. Each field
you release cuts the population into finer slices, and with 100 patients you run
out of people very quickly.

Now try adding the site as well.

In [ ]:
# RUN. Add site on top of district.
_ = k_check(safe, ["sex", "age_band", "postcode_district", "redcap_data_access_group"], k=5)

**Interpret.** Adding site changed nothing at all. Why?

Because each site draws from its own local districts, so knowing the district
already tells you the site. The field was **redundant** — it carried no
information you did not already have.

That is worth holding onto: the number of columns you release is not the same
as the amount of identifying information you release.

**YOUR TURN 3.**

1. In your own words, why does adding an indirect identifier make things worse so
   quickly?
2. You need every group to have at least 5 people. Get there. Options: wider age
   bands, coarser geography, dropping a field entirely, or suppressing the rows
   that fail.
3. For whichever route you chose, say **what analysis you can no longer do** as a
   result. Be specific: name a question a researcher would want to ask that your
   released dataset can no longer answer.
4. Would your answer change if this data were going into a Trusted Research
   Environment rather than being published as an open file? Why?

That last part is the whole exercise. Anyone can make a dataset anonymous by
deleting it.

In [ ]:
# YOUR TURN 3. Experiment here. Try wider age bands, coarser postcodes,
# dropping a field. Re-run k_check each time.

# Example: postcode area rather than district (the letters only)
safe["postcode_area"] = safe["postcode_district"].str.replace(r"\d+", "", regex=True)

_ = k_check(safe, ["sex", "age_band", "postcode_area"], k=5)

---
# Part 4 — The field everyone forgets

**READ.** We have done nothing to the free-text columns. Look at them.

In [ ]:
# RUN. The clinician comments. We drop the blanks and the repeats so that
# what is left is the distinct set of things people actually wrote.
comments = df["clinician_comments_txt"].fillna("")
comments = comments[comments.str.strip() != ""].drop_duplicates()

print(f"{len(comments)} distinct comments\n")
for c in comments:
    print("-", c)

**Interpret.** Count what you can see in there: next of kin names, telephone
numbers, a street address, named clinicians, a date of birth restated in prose, a
hospital number from an old system, an interpreter language.

None of this is in any structured field. All of it is identifying.

In [ ]:
# RUN. An automated attempt: find phone numbers.
import re

PHONE = re.compile(r"\b0\d{3}\s?\d{3}\s?\d{3,4}\b|\b07\d{3}\s?\d{6}\b")

redacted = comments.str.replace(PHONE, "[PHONE]", regex=True)

# Show only the comments that mention a person, so the remaining problem is visible
interesting = redacted[redacted.str.contains(r"\b(?:Dr|Mrs|Mr|Ms|Rev)\b|wife|son|daughter|brother|neighbour|employee|midwife", regex=True)]
for c in interesting:
    print("-", c)

**YOUR TURN 4 — the important question.**

The regular expression above catches phone numbers. Read the redacted output and
find what is still there.

1. List three identifying things the regex did not catch.
2. Two comments describe the patient's occupation. Find them. Why is
   "retired midwife at this hospital" more identifying than a phone number, in a
   dataset of 100 patients across 3 sites?
3. Could you write a rule that catches it? Could a language model? What would you
   have to accept about the reliability of either?
4. Given all that, what would you actually do with this column before release?

---
# Part 5 — The password that would not go away

**READ.** Research code ends up in git repositories. Sooner or later somebody puts
a database password in a script, notices, and deletes it. This section shows you
why deleting it does not help.

The cell below builds a tiny git repository inside this session, with two
commits: one that adds a script containing a password, and one that removes it.
Nothing here touches anything outside this notebook.

In [ ]:
# RUN. Build a small git repository with a mistake in its history.
import os, subprocess, textwrap

REPO = os.path.join(os.getcwd(), "northstar-analysis")

def git(cmd):
    return subprocess.run("git " + cmd, shell=True, cwd=REPO, capture_output=True, text=True).stdout

os.makedirs(REPO, exist_ok=True)
subprocess.run("git init -q", shell=True, cwd=REPO)
git('config user.email "student@example.org"')
git('config user.name "A Student"')

# commit 1: the script, with a password hard-coded
with open(os.path.join(REPO, "analysis.py"), "w") as f:
    f.write(textwrap.dedent("""
        import pandas as pd
        DB_HOST = "northstar-db.example.nhs.uk"
        DB_USER = "analyst"
        DB_PASSWORD = "Nst4r!2025-prod"      # TODO move this somewhere safer
        def connect():
            return f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/northstar"
    """).lstrip())
git("add analysis.py")
git('commit -q -m "first pass at the analysis script"')

# commit 2: password removed, read from the environment instead
with open(os.path.join(REPO, "analysis.py"), "w") as f:
    f.write(textwrap.dedent("""
        import os
        import pandas as pd
        DB_HOST = "northstar-db.example.nhs.uk"
        DB_USER = "analyst"
        DB_PASSWORD = os.environ["NORTHSTAR_DB_PASSWORD"]
        def connect():
            return f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/northstar"
    """).lstrip())
git("add analysis.py")
git('commit -q -m "remove password from code"')

print(git("log --oneline"))
print("current analysis.py:\n")
print(open(os.path.join(REPO, "analysis.py")).read())

**YOUR TURN 5.** The password is not in the current file. Is it gone?

Run the cell below. `git log -p` shows every change ever made, including what was
removed.

In [ ]:
# RUN. Look at the full history.
print(git("log -p --no-color"))

**YOUR TURN 5, continued.** Answer in the cell below.

1. Where is the password now? Who can read it?
2. The developer pushed this repository to GitHub between the two commits. What
   is the correct response, in order of urgency? (Hint: the first step is not
   about git at all.)
3. The second commit reads the password from an environment variable. Where does
   that variable's value live, and what file must never be committed?
4. Suppose the repository had never been pushed anywhere. Is there a way to make
   the password disappear from history? Would you trust it?

*Your answer here:*

---
# Part 6 — Your verdict

**YOUR TURN 6.** In the cell below, write six to ten sentences answering:

> A clinical colleague asks for a copy of this dataset on their laptop so they
> can start modelling. **What do you tell them?**

Structure it as: what in the data could identify a patient, what you did about
it, what that cost the analysis, what you would refuse to release in any form,
and what setting you would insist on instead of a laptop.

You are not being asked to say no. You are being asked to say what has to be true
first.

*Your answer here:*